# Build the all-MiniLM-L6-v2 sentence embedder for LiteRT

Replaces the Week 3 MediaPipe `bert_embedder`, whose raw BERT pooler output did not separate related from unrelated sentences.

| File | Contract |
| --- | --- |
| `minilm_l6_text.tflite` | input `int32[1, 128]` WordPiece IDs padded with 0 → `float32[1, 384]`, mean-pooled and L2-normalized inside the graph |
| `minilm_l6_vocab.txt` | uncased WordPiece vocabulary (30522 tokens) |

Run in Google Colab (CPU is enough): **Runtime → Run all**. The last cell downloads both files; put them in the project's `models/` folder and run `python tools/verify_models.py`.

In [ ]:
%%bash
pip install -q litert-torch transformers sentence-transformers
# litert-torch may downgrade typing_extensions; keep the version Colab needs.
pip install -q -U "typing_extensions>=4.16"

In [ ]:
import hashlib
import shutil

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CONTEXT_LENGTH = 128

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, attn_implementation="eager").eval()


class SentenceEncoderExport(torch.nn.Module):
    """Token IDs -> mean-pooled, L2-normalized sentence embedding.

    The attention mask is derived from the IDs (padding ID 0), so the
    runtime only has to pass one tensor.
    """

    def __init__(self, model: torch.nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        ids = token_ids.to(torch.long)
        mask = (ids != 0).to(torch.long)
        hidden = self.model(
            input_ids=ids,
            attention_mask=mask,
            token_type_ids=torch.zeros_like(ids),
            return_dict=True,
        ).last_hidden_state
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1e-9)
        return torch.nn.functional.normalize(pooled, p=2, dim=1)


module = SentenceEncoderExport(encoder).eval()
(vocab_file,) = hf_tokenizer.save_vocabulary(".")
shutil.move(vocab_file, "minilm_l6_vocab.txt")
print("vocabulary size:", len(open("minilm_l6_vocab.txt", encoding="utf-8").read().splitlines()))

In [ ]:
from tokenizers import BertWordPieceTokenizer

# Same tokenizer class and settings as the project's BertTokenizer.
runtime_tokenizer = BertWordPieceTokenizer("minilm_l6_vocab.txt", lowercase=True)
runtime_tokenizer.enable_truncation(max_length=CONTEXT_LENGTH)


def runtime_ids(text: str) -> np.ndarray:
    ids = runtime_tokenizer.encode(text).ids
    padded = np.zeros((1, CONTEXT_LENGTH), dtype=np.int32)
    padded[0, : len(ids)] = ids
    return padded


checks = [
    "How do I make Tomato Soup?",
    "Café menus, naïve résumé: ACCENTS & punctuation!",
    "PDFium extracts text; 3.5 MB file.",
    "machine learning model inference on a phone",
]
for text in checks:
    expected = hf_tokenizer(text, truncation=True, max_length=CONTEXT_LENGTH)["input_ids"]
    actual = runtime_tokenizer.encode(text).ids
    print("match    " if actual == expected else "DIFFERENT", repr(text))

In [ ]:
from sentence_transformers import SentenceTransformer

reference_model = SentenceTransformer(MODEL_NAME, device="cpu")
sentences = checks + [
    "Tomato soup recipe: cook tomatoes with garlic and basil, then blend.",
    "This project searches local documents without sending data to the cloud.",
]
reference = reference_model.encode(sentences, normalize_embeddings=True)

with torch.no_grad():
    wrapped = np.concatenate(
        [module(torch.from_numpy(runtime_ids(text))).numpy() for text in sentences]
    )

agreement = (wrapped * reference).sum(axis=1)
print("wrapper vs sentence-transformers cosine:", np.round(agreement, 6))
assert agreement.min() > 0.9999, "Export wrapper does not reproduce the reference model"

In [ ]:
import litert_torch

sample = torch.from_numpy(runtime_ids("a sample sentence for tracing"))
edge_model = litert_torch.convert(module, (sample,))
edge_model.export("minilm_l6_text.tflite")
print("Exported minilm_l6_text.tflite")

In [ ]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path="minilm_l6_text.tflite")
interpreter.allocate_tensors()
inputs = interpreter.get_input_details()
outputs = interpreter.get_output_details()
print("inputs: ", [(d["name"], tuple(d["shape"]), np.dtype(d["dtype"]).name) for d in inputs])
print("outputs:", [(d["name"], tuple(d["shape"]), np.dtype(d["dtype"]).name) for d in outputs])

assert len(inputs) == 1 and tuple(inputs[0]["shape"]) == (1, CONTEXT_LENGTH)
assert np.dtype(inputs[0]["dtype"]) == np.int32
assert len(outputs) == 1 and tuple(outputs[0]["shape"]) == (1, 384)


def litert_embed(text: str) -> np.ndarray:
    interpreter.set_tensor(inputs[0]["index"], runtime_ids(text))
    interpreter.invoke()
    return np.asarray(interpreter.get_tensor(outputs[0]["index"])[0], dtype=np.float32)


litert = np.stack([litert_embed(text) for text in sentences])
for text, a, b in zip(sentences, litert, reference):
    print(f"cosine {float(a @ b):.6f}  max abs error {np.abs(a - b).max():.2e}  {text!r}")
assert (litert * reference).sum(axis=1).min() > 0.9999, "LiteRT parity check failed"

In [ ]:
# The same five checks as tools/diagnose_text_embeddings.py.
TRIPLES = [
    ("how do I make tomato soup",
     "Tomato soup recipe: cook tomatoes with garlic and basil, then blend.",
     "This project searches local documents without sending data to the cloud."),
    ("The cat sat on the mat.", "A kitten is resting on a rug.",
     "Quarterly revenue grew by ten percent."),
    ("Where are my tax documents?", "Locating last year's tax return files.",
     "The football match ended in a draw."),
    ("searching private files offline", "Local search that keeps documents on the device.",
     "Slow cooked beef stew with carrots."),
    ("machine learning model inference on a phone", "Running neural networks on mobile devices.",
     "Gardening tips for growing roses in spring."),
]

margins = []
for anchor, related, unrelated in TRIPLES:
    a, r, u = (litert_embed(text) for text in (anchor, related, unrelated))
    margins.append(float(a @ r - a @ u))
    print(f"related {float(a @ r):.3f}  unrelated {float(a @ u):.3f}  margin {margins[-1]:.3f}  {anchor}")

correct = sum(margin > 0 for margin in margins)
print(f"ordering correct: {correct}/{len(TRIPLES)}, mean margin {np.mean(margins):.3f}")
assert correct == len(TRIPLES), "Semantic sanity check failed"

In [ ]:
from google.colab import files

for name in ["minilm_l6_text.tflite", "minilm_l6_vocab.txt"]:
    with open(name, "rb") as handle:
        data = handle.read()
    print(f"{name:24} {len(data) / (1024 * 1024):6.1f} MB  {hashlib.sha256(data).hexdigest()}")
    files.download(name)